First Test of CIM Error Bound (ReRAM Crossbars)

In [9]:
import numpy as np
from scipy.optimize import root
import time

# 1. SETUP SIMULATION PARAMETERS
np.random.seed(42)  
N = 512            # 8x8 Crossbar
alpha = 2.0         # RRAM non-linearity factor (V^-1)
I0_nominal = 1e-5   # Nominal scaling current (10 uA)
RL = 10000.0        # Load resistance at the bottom of columns (10 kOhm)
sigma_mismatch = 0.2  # 5% device-to-device fabrication variation

# Input Activation Vector (High voltages to provoke non-linearity)
rng = np.random.default_rng()

V_in = rng.random(N)/2
V_max = np.max(V_in)

# Generate nominal and perturbed device matrices
I0_matrix = np.full((N, N), I0_nominal)
Delta_I0 = np.random.normal(0, sigma_mismatch * I0_nominal, (N, N))
I0_perturbed = I0_matrix + Delta_I0

# 2. EXACT NON-LINEAR SOLVER (GROUND TRUTH SPICE DESIGN)
def spice_kcl_residual(V_col):
    residual = np.zeros(N)
    for j in range(N):
        current_from_rows = 0.0
        for i in range(N):
            V_drop = V_in[i] - V_col[j]
            current_from_rows += I0_perturbed[i, j] * np.sinh(alpha * V_drop)
        residual[j] = current_from_rows - (V_col[j] / RL)
    return residual
start_time_spice = time.perf_counter()


spice_solution = root(spice_kcl_residual, x0=np.zeros(N))
V_col_spice = spice_solution.x

# 3. IDEAL LINEAR SOLVER (RTL ACCELERATOR BASELINE)
g0_constant = alpha * I0_nominal
A0_diag = np.zeros(N)
B0_vector = np.zeros(N)
for j in range(N):
    A0_diag[j] = (1.0 / RL) + (N * g0_constant)
    B0_vector[j] = np.sum(g0_constant * V_in)

V_col_ideal = B0_vector / A0_diag

# Calculate true empirical error vector norm
e_empirical = np.max(np.abs(V_col_spice - V_col_ideal))

end_time_spice = time.perf_counter()

start_time_bound = time.perf_counter()

# 4. RIGOROUS BOUND EVALUATION (CORRECTED)
# Evaluate the true non-linear residual vector at the ideal voltage vector
F_true_at_V_ideal = np.zeros(N)
J0_diag = np.zeros(N)
Delta_J_diag = np.zeros(N)
H_max_j = np.zeros(N)

for j in range(N):
    # Compute true KCL check at V_ideal
    current_in = 0.0
    for i in range(N):
        V_drop_ideal = V_in[i] - V_col_ideal[j]
        current_in += I0_perturbed[i, j] * np.sinh(alpha * V_drop_ideal)
    F_true_at_V_ideal[j] = current_in - (V_col_ideal[j] / RL)
    
    # Compute local Jacobian elements at this exact operating point
    for i in range(N):
        V_drop_ideal = V_in[i] - V_col_ideal[j]
        cosh_factor = np.cosh(alpha * V_drop_ideal)
        
        J0_diag[j] += alpha * I0_nominal * cosh_factor
        Delta_J_diag[j] += alpha * np.abs(Delta_I0[i, j]) * cosh_factor
        
        # Max local Hessian calculation
        H_max_j[j] += (alpha**2) * I0_perturbed[i, j] * np.sinh(alpha * V_max)
    
    J0_diag[j] += (1.0 / RL)

# Extract Operator Norms
inv_J0_norm = np.max(1.0 / J0_diag)

# Re-calculate corrected quadratic coefficients
C_linear = 1.0 - (inv_J0_norm * np.max(Delta_J_diag))
C_1 = inv_J0_norm * np.max(np.abs(F_true_at_V_ideal))
C_2 = 0.5 * inv_J0_norm * np.max(H_max_j)

# Compute strict upper bound
discriminant = (C_linear**2) - (4.0 * C_1 * C_2)

if discriminant >= 0 and C_linear > 0:
    e_bound = (C_linear - np.sqrt(discriminant)) / (2.0 * C_2)
    conservatism_index = e_bound / e_empirical
else:
    e_bound = np.nan
    conservatism_index = np.nan

end_time_bound = time.perf_counter()

# 5. DISPLAY VERIFICATION REPORT
print("====================================================")
print("       CORRECTED PRE-SILICON VALIDATION REPORT      ")
print("====================================================")
print(f"Max Applied Input Voltage (V_max) : {V_max:.3f} V")
print(f"Device-to-Device Variation       : {sigma_mismatch*100:.1f}%")
print("----------------------------------------------------")
print(f"True SPICE Voltage Error (Norm)   : {e_empirical:.6f} V")
print(f"Theoretical Quadratic Bound       : {e_bound:.6f} V")
print("----------------------------------------------------")
if not np.isnan(e_bound):
    print(f"Conservatism Index (Pessimism)    : {conservatism_index:.3f}x")
    if conservatism_index >= 1.0:
        print("SUCCESS: Bound is securely conservative.")
    else:
        print("CRITICAL FAILURE: Bound is still leaking.")
else:
    print("HAZARD WARNING: Contraction boundary collapsed.")
print("----------------------------------------------------")
print(f"True SPICE Error Computation Time : {-(start_time_spice - end_time_spice)} sec")
print(f"Quadratic Bound Computation Time  : {-(start_time_bound - end_time_bound)} sec")
print("----------------------------------------------------")
print("====================================================")

       CORRECTED PRE-SILICON VALIDATION REPORT      
Max Applied Input Voltage (V_max) : 0.499 V
Device-to-Device Variation       : 20.0%
----------------------------------------------------
True SPICE Voltage Error (Norm)   : 0.003846 V
Theoretical Quadratic Bound       : 0.004672 V
----------------------------------------------------
Conservatism Index (Pessimism)    : 1.215x
SUCCESS: Bound is securely conservative.
----------------------------------------------------
True SPICE Error Computation Time : 113.84117609998793 sec
Quadratic Bound Computation Time  : 1.821236100018723 sec
----------------------------------------------------


Next Testbench: Capacitive Crossbars

In [ ]:
import numpy as np
from scipy.optimize import root

# 1. SETUP CAPACITIVE CIM PARAMETERS
np.random.seed(101)
N = 8               # 8x8 Capacitive Matrix
C0_nominal = 1e-12  # Baseline capacitance (1 pF)
beta_nominal = 2e-12 # Cubic non-linearity coefficient (F/V^2)
sigma_mismatch = 0.04 # 4% fabrication variance in capacitor area

# Dynamic Input Activations
V_in = rng.random(N)/2
V_max = np.max(V_in)

# Generate perturbed capacitor fields
C0_perturbed = np.random.normal(C0_nominal, sigma_mismatch * C0_nominal, (N, N))
beta_perturbed = np.random.normal(beta_nominal, sigma_mismatch * beta_nominal, (N, N))

# 2. GROUND TRUTH SPICE-LEVEL CHARGE CONSERVATION SOLVER
def capacitive_kcl_residual(V_out):
    """
    Evaluates charge balance at the output sharing nodes.
    Equation: Q = C0*(Vin - Vout) + beta*(Vin - Vout)^3
    """
    residual = np.zeros(N)
    for j in range(N):
        total_charge = 0.0
        for i in range(N):
            V_diff = V_in[i] - V_out[j]
            total_charge += C0_perturbed[i, j] * V_diff + beta_perturbed[i, j] * (V_diff ** 3)
        residual[j] = total_charge
    return residual

# Run exact non-linear root finder
spice_solution = root(capacitive_kcl_residual, x0=np.zeros(N))
V_out_spice = spice_solution.x

# 3. REFERENCE MODEL SOLVER (IDEAL LINEAR BEHAVIORAL MODEL)
V_out_ideal = np.zeros(N)
for j in range(N):
    # Linear charge division: Vout = sum(C0*Vin) / sum(C0)
    V_out_ideal[j] = np.sum(C0_nominal * V_in) / (N * C0_nominal)

e_empirical = np.max(np.abs(V_out_spice - V_out_ideal))

# 4. EVALUATE LAYOUT-ABSTRACTED BOUNDS ON CHARGE DOMAIN
J0_diag = np.zeros(N)
Delta_J_diag = np.zeros(N)
F_true_at_V_ideal = np.zeros(N)
H_max_j = np.zeros(N)

for j in range(N):
    # True non-linear charge accumulation evaluated at the ideal operating point
    total_charge_ideal = 0.0
    for i in range(N):
        V_diff_ideal = V_in[i] - V_out_ideal[j]
        total_charge_ideal += C0_perturbed[i, j] * V_diff_ideal + beta_perturbed[i, j] * (V_diff_ideal ** 3)
        
        # Linear Jacobian component: dQ/dV = C0 + 3*beta*(V_diff)^2
        jacobian_element_nominal = C0_nominal + 3.0 * beta_nominal * (V_diff_ideal ** 2)
        jacobian_element_perturbed = C0_perturbed[i, j] + 3.0 * beta_perturbed[i, j] * (V_diff_ideal ** 2)
        
        J0_diag[j] += jacobian_element_nominal
        Delta_J_diag[j] += np.abs(jacobian_element_perturbed - jacobian_element_nominal)
        
        # Capacitive Curvature: d^2Q/dV^2 = 6 * beta * V_diff
        H_max_j[j] += 6.0 * beta_perturbed[i, j] * np.abs(V_max)
        
    F_true_at_V_ideal[j] = total_charge_ideal

# Invert operator bounds via L_infinity norm
inv_J0_norm = np.max(1.0 / J0_diag)

# Assign Unified Quadratic Parameters
C_linear = 1.0 - (inv_J0_norm * np.max(Delta_J_diag))
C_1 = inv_J0_norm * np.max(np.abs(F_true_at_V_ideal))
C_2 = 0.5 * inv_J0_norm * np.max(H_max_j)

# Extract error tracking root
discriminant = (C_linear ** 2) - (4.0 * C_1 * C_2)
if discriminant >= 0 and C_linear > 0:
    e_bound = (C_linear - np.sqrt(discriminant)) / (2.0 * C_2)
    conservatism_index = e_bound / e_empirical
else:
    e_bound = np.nan
    conservatism_index = np.nan

# 5. GENERATE ARCHITECTURE ASSESSMENT REPORT
print("====================================================")
print("     CHARGE-DOMAIN (CAPACITIVE) VALIDATION REPORT    ")
print("====================================================")
print(f"Max Operating Voltage Target     : {V_max:.3f} V")
print(f"Capacitor Area Tolerance (Sigma) : {sigma_mismatch*100:.1f}%")
print("----------------------------------------------------")
print(f"True Exact Charge Error (Norm)   : {e_empirical:.6f} V")
print(f"Theoretical Closed Bound         : {e_bound:.6f} V")
print("----------------------------------------------------")
if not np.isnan(e_bound):
    print(f"Conservatism Index (Pessimism)    : {conservatism_index:.3f}x")
    if conservatism_index >= 1.0:
        print("SUCCESS: General framework bounds capacitive architecture cleanly.")
    else:
        print("CRITICAL VIOLATION: Bound underpredicted charge drift.")
else:
    print("HAZARD: Switched capacitor tracking bounds collapsed.")
print("====================================================")

     CHARGE-DOMAIN (CAPACITIVE) VALIDATION REPORT    
Max Operating Voltage Target     : 0.400 V
Capacitor Area Tolerance (Sigma) : 4.0%
----------------------------------------------------
True Exact Charge Error (Norm)   : 0.003144 V
Theoretical Closed Bound         : 0.003325 V
----------------------------------------------------
Conservatism Index (Pessimism)    : 1.058x
SUCCESS: General framework bounds capacitive architecture cleanly.


Now, let's add time dependence into the simulations

In [6]:
import numpy as np
from scipy.optimize import root

# 1. HARDWARE ENVIRONMENT CONFIGURATION
np.random.seed(2026)  
N = 128                # 8x8 Crossbar
alpha = 2.5          # RRAM non-linearity factor (V^-1)
I0_nominal = 1.2e-5  # Nominal scaling current (12 uA)
RL = 8000.0          # Load resistance (8 kOhm)
sigma_mismatch = 0.05 # 5% device variation

# Time Configuration
num_cycles = 6
time_steps = np.arange(num_cycles)

# Generate static random manufacturing variance
I0_matrix = np.full((N, N), I0_nominal)
Delta_I0 = np.random.normal(0, sigma_mismatch * I0_nominal, (N, N))
I0_perturbed = I0_matrix + Delta_I0

# 2. DEFINITION OF KCL NETLIST ENGINES
def spice_kcl_residual(V_col, V_in_t):
    """Exact non-linear KCL solver matching physical SPICE loop."""
    residual = np.zeros(N)
    for j in range(N):
        current_in = 0.0
        for i in range(N):
            V_drop = V_in_t[i] - V_col[j]
            current_in += I0_perturbed[i, j] * np.sinh(alpha * V_drop)
        residual[j] = current_in - (V_col[j] / RL)
    return residual

# 3. TRANSIENT SIMULATION TRACKING LOOP
print("=========================================================================")
print("             DYNAMIC TIME-STEP PRE-SILICON ACCELERATOR REPORT            ")
print("=========================================================================")
print(f"{'Cycle':<7}{'Max V_in':<12}{'True Error':<16}{'Bound Fence':<16}{'Pessimism (ρ)':<12}")
print("-------------------------------------------------------------------------")

for t in range(num_cycles):
    # Dynamically generate a time-varying input activation vector (Transient signal)
    # Simulates varying neural network payload pulses over time
    signal_amplitude = 0.35 + 0.15 * np.sin(2 * np.pi * t / num_cycles)
    V_in_t = np.array([signal_amplitude - (0.04 * i) for i in range(N)])
    V_in_t = np.clip(V_in_t, 0.0, 0.6) # Clamp to legal hardware thresholds
    V_max_t = np.max(V_in_t)
    
    # Track exact SPICE voltages for current cycle
    spice_solution = root(spice_kcl_residual, x0=np.zeros(N), args=(V_in_t,))
    V_col_spice = spice_solution.x
    
    # Compute ideal linear behavior reference (RTL baseline comparison)
    g0_constant = alpha * I0_nominal
    A0_diag = np.zeros(N)
    B0_vector = np.zeros(N)
    for j in range(N):
        A0_diag[j] = (1.0 / RL) + (N * g0_constant)
        B0_vector[j] = np.sum(g0_constant * V_in_t)
    V_col_ideal = B0_vector / A0_diag
    
    # Calculate ground-truth empirical error norm
    e_empirical = np.max(np.abs(V_col_spice - V_col_ideal))
    
    # 4. EVALUATE DYNAMIC O(1) BOUND FOR TIME 't'
    F_true_at_V_ideal = np.zeros(N)
    J0_diag = np.zeros(N)
    Delta_J_diag = np.zeros(N)
    H_max_j = np.zeros(N)
    
    for j in range(N):
        current_in_ideal = 0.0
        for i in range(N):
            V_drop_ideal = V_in_t[i] - V_col_ideal[j]
            current_in_ideal += I0_perturbed[i, j] * np.sinh(alpha * V_drop_ideal)
            
            # Update Jacobian and bounds with current cycle's operating state
            cosh_factor = np.cosh(alpha * V_drop_ideal)
            J0_diag[j] += alpha * I0_nominal * cosh_factor
            Delta_J_diag[j] += alpha * np.abs(Delta_I0[i, j]) * cosh_factor
            
            # Dynamic Curvature Lookup modification: γ(t) embedded in local Hessian
            H_max_j[j] += (alpha**2) * I0_perturbed[i, j] * np.sinh(alpha * V_max_t)
            
        J0_diag[j] += (1.0 / RL)
        F_true_at_V_ideal[j] = current_in_ideal - (V_col_ideal[j] / RL)
        
    inv_J0_norm = np.max(1.0 / J0_diag)
    
    # Compute cycle-specific boundary factors
    C_linear = 1.0 - (inv_J0_norm * np.max(Delta_J_diag))
    C_1 = inv_J0_norm * np.max(np.abs(F_true_at_V_ideal))
    C_2 = 0.5 * inv_J0_norm * np.max(H_max_j)
    
    # Evaluate discriminant track
    discriminant = (C_linear**2) - (4.0 * C_1 * C_2)
    
    if discriminant >= 0 and C_linear > 0:
        e_bound = (C_linear - np.sqrt(discriminant)) / (2.0 * C_2)
        rho = e_bound / e_empirical
        rho_str = f"{rho:.3f}x"
        bound_str = f"{e_bound:.6f} V"
    else:
        bound_str = "UNSTABLE"
        rho_str = "INF"
        
    print(f"t={t:<5}{V_max_t:<12.3f}{e_empirical:<16.6f}{bound_str:<16}{rho_str:<12}")

print("=========================================================================")

             DYNAMIC TIME-STEP PRE-SILICON ACCELERATOR REPORT            
Cycle  Max V_in    True Error      Bound Fence     Pessimism (ρ)
-------------------------------------------------------------------------
t=0    0.350       0.001445        0.001529 V      1.058x      
t=1    0.480       0.003651        0.003859 V      1.057x      
t=2    0.480       0.003651        0.003859 V      1.057x      
t=3    0.350       0.001445        0.001529 V      1.058x      
t=4    0.220       0.000469        0.000492 V      1.049x      
t=5    0.220       0.000469        0.000492 V      1.049x      


Now that we've proved the Bound's general efficacy, let's try cleaning the code up

In [ ]:
import numpy as np
from scipy.optimize import root
import time

# 1. HARDWARE ENVIRONMENT CONFIGURATION
np.random.seed(124346)  
N = 128            # 128x128 Crossbar Array
alpha = 2.5          # RRAM non-linearity factor (V^-1)
I0_nominal = 1.2e-5  # Nominal scaling current (12 uA)
RL = 8000.0          # Load resistance (8 kOhm)
sigma_mismatch = 0.4  # 20% device variation (Aggressive mismatch corner)

num_cycles = 15

# --- FOUNDRY STATISTICAL CORNER KNOWLEDGE ---
kappa = 3  # 5-Sigma confidence multiplier for high-yield tracking
I0_worst_case = I0_nominal + (kappa * sigma_mismatch * I0_nominal)

# --- PHYSICAL CHIP INSTANCE (HIDDEN FROM THE OBSERVER) ---
I0_perturbed = np.random.normal(I0_nominal, sigma_mismatch * I0_nominal, (N, N))

# COMPILE-TIME CONFIGURATION (Verifier-Accessible Invariants)
g0_constant = alpha * I0_nominal
J0_diag_vals = np.full(N, (1.0 / RL) + (N * g0_constant))
inv_J0_norm = np.max(1.0 / J0_diag_vals)

# Statistical C_linear calculation leveraging RSS variance reduction
C_linear = 1.0 - (inv_J0_norm * np.sqrt(N) * alpha * (sigma_mismatch * I0_nominal) * kappa)

def spice_kcl_residual(V_col, V_in_t):
    """Physical SPICE reference emulator (Iterative O(N^3) solver target)."""
    residual = np.zeros(N)
    for j in range(N):
        current_in = 0.0
        for i in range(N):
            V_drop = V_in_t[i] - V_col[j]
            current_in += I0_perturbed[i, j] * np.sinh(alpha * V_drop)
        residual[j] = current_in - (V_col[j] / RL)
    return residual

# Time Profiling Accumulators
total_spice_time = 0.0
total_observer_time = 0.0

print("=========================================================================")
# Print heading modified to reflect your actual configuration profile
print(f"       AGGRESSIVE O(N^2) STATISTICAL RSS PRE-SILICON PROFILE REPORT      ")
print("=========================================================================")
print(f"{'Cycle':<7}{'Max V_in':<12}{'True Error':<16}{'Bound Fence':<16}{'Pessimism (ρ)':<12}")
print("-------------------------------------------------------------------------")

rng = np.random.default_rng()

for t in range(num_cycles):
    signal_amplitude = rng.random()*0.5
    V_in_t = np.array([signal_amplitude - (0.04 * i) for i in range(N)])
    V_in_t = np.clip(V_in_t, 0.0, 1.0) 
    alpha_t = np.max(V_in_t)
    
    # ---------------------------------------------------------------------
    # TIMED PASS 1: SPICE ITERATIVE SOLUTION
    # ---------------------------------------------------------------------
    t_spice_start = time.perf_counter()
    spice_solution = root(spice_kcl_residual, x0=np.zeros(N), args=(V_in_t,))
    V_col_spice = spice_solution.x
    t_spice_end = time.perf_counter()
    total_spice_time += (t_spice_end - t_spice_start)
    
    # Compute baseline reference values
    V_col_ideal = np.full(N, np.sum(g0_constant * V_in_t)) / J0_diag_vals
    e_empirical = np.max(np.abs(V_col_spice - V_col_ideal))

    # =====================================================================
    # TIMED PASS 2: VERIFIER-BLIND HIGH-AGGRESSION RSS OBSERVER 
    # =====================================================================
    t_obs_start = time.perf_counter()
    
    F_bounded_at_V_ideal = np.zeros(N)
    
    
    for j in range(N):
        current_nominal = 0.0
        sum_sinh_sq = 0.0  
        sum_skew_correction = 0.0 # Captures higher-order probability stretching
        
        for i in range(N):
            V_drop_ideal = V_in_t[i] - V_col_ideal[j]
            sinh_val = np.sinh(alpha * V_drop_ideal)
            cosh_val = np.cosh(alpha * V_drop_ideal)
            
            current_nominal += I0_nominal * sinh_val
            sum_sinh_sq += sinh_val ** 2  
            
            # Mathematical Skewness Tracking: 
            # Intersects crossbar voltage drops with variation kurtosis exponents
            sum_skew_correction += (alpha * sinh_val * cosh_val) ** 2
            
        F_nominal_j = current_nominal - (V_col_ideal[j] / RL)
        
        # Base linear standard deviation
        sigma_linear = (sigma_mismatch * I0_nominal) * np.sqrt(sum_sinh_sq)
        
        # Dynamic skew correction factor (prevents under-bounding at high voltages)
        sigma_skew = (sigma_mismatch * I0_nominal)**2 * np.sqrt(0.5 * sum_skew_correction)
        
        # Complete non-linear distribution envelope
        F_bounded_at_V_ideal[j] = np.abs(F_nominal_j) + (kappa * sigma_linear) + (kappa**2 * sigma_skew)
    
    
    C_1 = inv_J0_norm * np.max(F_bounded_at_V_ideal)
    
    # Uses the statistical bound corner instead of real-time cell sniffing
    gamma_t = (alpha**2) * I0_worst_case * np.sinh(alpha * alpha_t)
    C_2 = inv_J0_norm * (N * gamma_t)
    
    discriminant = (C_linear**2) - (4.0 * C_1 * C_2)
    
    if discriminant >= 0 and C_linear > 0:
        e_bound = (C_linear - np.sqrt(discriminant)) / (2.0 * C_2)
        rho = e_bound / e_empirical
        rho_str = f"{rho:.3f}x"
        bound_str = f"{e_bound:.6f} V"
    else:
        bound_str = "HAZARD"
        rho_str = " N/A"
        
    t_obs_end = time.perf_counter()
    total_observer_time += (t_obs_end - t_obs_start)
    # =====================================================================
        
    print(f"t={t:<5}{alpha_t:<12.3f}{e_empirical:<16.6f}{bound_str:<16}{rho_str:<12}")

print("-------------------------------------------------------------------------")
print(f"... Completed transient evaluation across {num_cycles} contiguous clock phases.")
print("=========================================================================")
print("                      EDA SPEED PERFORMANCE REPORT                       ")
print("=========================================================================")
print(f"Total Iterative SPICE Engine Time : {total_spice_time:.5f} seconds")
print(f"Total RSS Mathematical Observer    : {total_observer_time:.5f} seconds")
print("-------------------------------------------------------------------------")
print(f"Absolute Validation Time Saved    : {total_spice_time - total_observer_time:.4f} seconds")
print(f"Pre-Silicon Verification Speedup   : {total_spice_time / total_observer_time:.2f}x faster")
print("=========================================================================")

       AGGRESSIVE O(N^2) STATISTICAL RSS PRE-SILICON PROFILE REPORT      
Cycle  Max V_in    True Error      Bound Fence     Pessimism (ρ)
-------------------------------------------------------------------------
t=0    0.346       0.006260        0.008001 V      1.278x      
t=1    0.466       0.010441        0.015503 V      1.485x      
t=2    0.494       0.011897        0.018318 V      1.540x      
t=3    0.114       0.000978        0.001440 V      1.473x      
t=4    0.150       0.001430        0.002098 V      1.467x      
t=5    0.208       0.002504        0.003373 V      1.347x      
t=6    0.487       0.011543        0.017564 V      1.522x      
t=7    0.213       0.002644        0.003507 V      1.326x      
t=8    0.243       0.003450        0.004326 V      1.254x      
t=9    0.318       0.005435        0.006830 V      1.257x      
t=10   0.221       0.002852        0.003710 V      1.301x      
t=11   0.444       0.009348        0.013719 V      1.468x      
t=12   0.121       

: 

In [ ]:
import numpy as np
from scipy.optimize import root
import time

# 1. HARDWARE ENVIRONMENT CONFIGURATION (CAPACITIVE DOMAIN)
np.random.seed(2026)  
N = 128               # 128x128 Capacitive Crossbar Array
C_nominal = 10e-15    # Nominal unit capacitance (10 fF MIM Cap)
C_para = 400e-15      # Column parasitic readout capacitance (400 fF)
lambda_nl = 0.4       # Voltage coefficient of capacitance (V^-2 non-linearity)
sigma_mismatch = 0.05 # 5% global capacitance manufacturing variation

num_cycles = 15

# --- FOUNDRY STATISTICAL CORNER KNOWLEDGE ---
kappa = 4.0  # 4-Sigma high-yield guardrail multiplier
C_worst_case = C_nominal + (kappa * sigma_mismatch * C_nominal)

# --- PHYSICAL DIE INSTANCE (HIDDEN FROM THE OBSERVER) ---
C_perturbed = np.random.normal(C_nominal, sigma_mismatch * C_nominal, (N, N))

# COMPILE-TIME CONFIGURATION (Verifier-Accessible Invariants)
# Nominal linear capacitance derivative baseline
g0_constant = C_nominal
J0_diag_vals = np.full(N, C_para + (N * g0_constant))
inv_J0_norm = np.max(1.0 / J0_diag_vals)

# Statistical C_linear attenuation for the charge domain
C_linear = 1.0 - (inv_J0_norm * np.sqrt(N) * (sigma_mismatch * C_nominal) * kappa)

def capacitive_charge_residual(V_col, V_in_t):
    """Physical charge conservation solver modeling the true hidden random die."""
    residual = np.zeros(N)
    for j in range(N):
        total_charge = 0.0
        for i in range(N):
            V_drop = V_in_t[i] - V_col[j]
            # Non-linear Charge-Voltage physical equation
            total_charge += C_perturbed[i, j] * V_drop * (1.0 + lambda_nl * (V_drop**2))
        # Net charge balance including the parasitic extraction capacitor
        residual[j] = total_charge - (C_para * V_col[j])
    return residual

# Profiling metrics
total_spice_time = 0.0
total_observer_time = 0.0

print("=========================================================================")
print("         CAPACITIVE CIM ACCELERATOR RSS PRE-SILICON PROFILE REPORT       ")
print("=========================================================================")
print(f"{'Cycle':<7}{'Max V_in':<12}{'True Error':<16}{'Bound Fence':<16}{'Pessimism (ρ)':<12}")
print("-------------------------------------------------------------------------")

for t in range(num_cycles):
    # Charge-injection voltage step profile
    signal_amplitude = 0.40 + 0.10 * np.sin(2 * np.pi * t / 6)
    V_in_t = np.array([signal_amplitude - (0.002 * i) for i in range(N)])
    V_in_t = np.clip(V_in_t, 0.0, 0.6) 
    alpha_t = np.max(V_in_t)
    
    # ---------------------------------------------------------------------
    # TIMED PASS 1: ITERATIVE CHARGE SOLVER (SPICE EQUIVALENT)
    # ---------------------------------------------------------------------
    t_spice_start = time.perf_counter()
    spice_solution = root(capacitive_charge_residual, x0=np.zeros(N), args=(V_in_t,))
    V_col_spice = spice_solution.x
    t_spice_end = time.perf_counter()
    total_spice_time += (t_spice_end - t_spice_start)
    
    # RTL Linear Charge Sharing Baseline
    V_col_ideal = np.full(N, np.sum(g0_constant * V_in_t)) / J0_diag_vals
    e_empirical = np.max(np.abs(V_col_spice - V_col_ideal))

    # =====================================================================
    # TIMED PASS 2: VERIFIER-BLIND CAPACITIVE RSS OBSERVER
    # =====================================================================
    t_obs_start = time.perf_counter()
    
    F_bounded_at_V_ideal = np.zeros(N)
    
    for j in range(N):
        charge_nominal = 0.0
        sum_normalized_charge_sq = 0.0
        sum_second_order_sq = 0.0  # Accumulator for the second-order curvature RSS term
        
        for i in range(N):
            V_drop_ideal = V_in_t[i] - V_col_ideal[j]
            
            # 1. First-Order Sensitivity: dQ / dC
            base_charge_term = V_drop_ideal * (1.0 + lambda_nl * (V_drop_ideal**2))
            
            # 2. Second-Order Cross-Sensitivity: d^2Q / (dC dV)
            # Captures how capacitance variation couples with non-linear voltage acceleration
            cross_curvature_term = 1.0 + 3.0 * lambda_nl * (V_drop_ideal**2)
            
            # Formulate nominal charge using baseline foundry properties
            charge_nominal += C_nominal * base_charge_term
            
            # Accumulate orthogonal components for the first- and second-order RSS bounds
            sum_normalized_charge_sq += base_charge_term ** 2
            sum_second_order_sq += cross_curvature_term ** 2
            
        F_nominal_j = charge_nominal - (C_para * V_col_ideal[j])
        
        # Pull first-order linear standard deviation
        sigma_node_j = (sigma_mismatch * C_nominal) * np.sqrt(sum_normalized_charge_sq)
        
        # Pull second-order non-linear skewness deviation 
        sigma_skew_j = (sigma_mismatch * C_nominal)**2 * np.sqrt(0.5 * sum_second_order_sq)
        
        # Final robust bound combining nominal drift, linear RSS, and second-order curvature RSS
        F_bounded_at_V_ideal[j] = np.abs(F_nominal_j) + (kappa * sigma_node_j) + (kappa**2 * sigma_skew_j)
        
    C_1 = inv_J0_norm * np.max(F_bounded_at_V_ideal)
    
    # Evaluate higher-order system acceleration via 2nd-order charge derivatives
    gamma_t = 6.0 * lambda_nl * C_worst_case * alpha_t
    C_2 = inv_J0_norm * (N * gamma_t)
    
    discriminant = (C_linear**2) - (4.0 * C_1 * C_2)
    
    if discriminant >= 0 and C_linear > 0:
        e_bound = (C_linear - np.sqrt(discriminant)) / (2.0 * C_2)
        rho = e_bound / e_empirical
        rho_str = f"{rho:.3f}x"
        bound_str = f"{e_bound:.6f} V"
    else:
        bound_str = "HAZARD TRIGGERED"
        rho_str = "INF"
        
    t_obs_end = time.perf_counter()
    total_observer_time += (t_obs_end - t_obs_start)
    # =====================================================================
        
    print(f"t={t:<5}{alpha_t:<12.3f}{e_empirical:<16.6f}{bound_str:<16}{rho_str:<12}")

print("-------------------------------------------------------------------------")
print("                      EDA SPEED PERFORMANCE REPORT                       ")
print("=========================================================================")
print(f"Total Iterative Charge Engine Time : {total_spice_time:.5f} seconds")
print(f"Total Capacitive RSS Observer Time : {total_observer_time:.5f} seconds")
print("-------------------------------------------------------------------------")
print(f"Absolute Validation Time Saved     : {total_spice_time - total_observer_time:.4f} seconds")
print(f"Pre-Silicon Verification Speedup    : {total_spice_time / total_observer_time:.2f}x faster")
print("=========================================================================")

         CAPACITIVE CIM ACCELERATOR RSS PRE-SILICON PROFILE REPORT       
Cycle  Max V_in    True Error      Bound Fence     Pessimism (ρ)
-------------------------------------------------------------------------
t=0    0.400       0.001209        0.001772 V      1.465x      
t=1    0.487       0.001612        0.002193 V      1.361x      
t=2    0.487       0.001612        0.002193 V      1.361x      
t=3    0.400       0.001209        0.001772 V      1.465x      
t=4    0.313       0.000892        0.001438 V      1.611x      
t=5    0.313       0.000892        0.001438 V      1.611x      
t=6    0.400       0.001209        0.001772 V      1.465x      
t=7    0.487       0.001612        0.002193 V      1.361x      
t=8    0.487       0.001612        0.002193 V      1.361x      
t=9    0.400       0.001209        0.001772 V      1.465x      
t=10   0.313       0.000892        0.001438 V      1.611x      
t=11   0.313       0.000892        0.001438 V      1.611x      
t=12   0.400       